In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import wbgapi as wb
from fredapi import Fred
from dotenv import load_dotenv
import os

load_dotenv()

print("All libraries loaded!")
print(f"pandas:   {pd.__version__}")
print(f"numpy:    {np.__version__}")
print(f"yfinance: {yf.__version__}")

All libraries loaded!
pandas:   3.0.0
numpy:    2.4.1
yfinance: 1.5.1


In [4]:
# WHY yfinance download for Nikkei:
# ^N225 is Yahoo Finance's ticker symbol for nikkie 225
# yfinance pulls historical daily prices directly - no manual

nikkei = yf.download('^N225', start='2014-01-01', end='2023-12-31')

print(f"Nikkei shape: {nikkei.shape}")
print(f"\nColumns: {nikkei.columns.tolist()}")
print(f"\nDate range: {nikkei.index.min()} to {nikkei.index.max()}")
nikkei.head()

[*********************100%***********************]  1 of 1 completed

Nikkei shape: (2444, 5)

Columns: [('Close', '^N225'), ('High', '^N225'), ('Low', '^N225'), ('Open', '^N225'), ('Volume', '^N225')]

Date range: 2014-01-06 00:00:00 to 2023-12-29 00:00:00


Price,Close,High,Low,Open,Volume
Ticker,^N225,^N225,^N225,^N225,^N225
Date,,,,,
2014-01-06,15908.879883,16164.009766,15864.440430,16147.540039,192700000
2014-01-07,15814.370117,15935.370117,15784.250000,15835.410156,165900000
2014-01-08,16121.450195,16121.450195,15906.570312,15943.679688,206700000
2014-01-09,15880.330078,16004.559570,15838.440430,16002.879883,217400000
2014-01-10,15912.059570,15922.139648,15754.700195,15785.150391,237500000


In [5]:
# WHY these two specific companies:
# Toyota = Japan's largest exporter, automotive sector proxy
# Sony = Japan's electronics/entertainment giant

toyota = yf.download('TM', start='2014-01-01', end='2023-12-31')
sony = yf.download('SONY', start='2014-01-01', end='2023-12-31')

print(f"Toyota shape: {toyota.shape}")
print(f"Sony shape: {sony.shape}")
print(f"\nToyota columns: {toyota.columns.tolist()}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Toyota shape: (2516, 5)
Sony shape: (2516, 5)

Toyota columns: [('Close', 'TM'), ('High', 'TM'), ('Low', 'TM'), ('Open', 'TM'), ('Volume', 'TM')]


In [6]:
# WHY flattern MultiIndex columns:
# Right now columns look like ('Close', '^N225') - a tuple
# When the time of merging stock data with GDP data (single-level)
# columns like 'gdp_usd'), mixing MultiIndex and single-level 
# columns in the same merge causes confusing KeyErrors.
# Flattening now means every DataFrame speaks the same

def flatten_columns(df, ticker_name):
    df = df.copy()
    df.columns = [col[0] for col in df.columns]
    df['ticker'] = ticker_name
    df = df.reset_index()
    return df

nikkei_clean = flatten_columns(nikkei, 'Nikkei225')
toyota_clean = flatten_columns(toyota, 'Toyota')
sony_clean = flatten_columns(sony, 'sony')

print("Nikkei columns after flatten:", nikkei_clean.columns.tolist())
print("\nSample:")
nikkei_clean.head(3)

Nikkei columns after flatten: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker']

Sample:


,Date,Close,High,Low,Open,Volume,ticker
0,2014-01-06,15908.879883,16164.009766,15864.440430,16147.540039,192700000,Nikkei225
1,2014-01-07,15814.370117,15935.370117,15784.250000,15835.410156,165900000,Nikkei225
2,2014-01-08,16121.450195,16121.450195,15906.570312,15943.679688,206700000,Nikkei225


In [7]:
# WHY save raw API pulls to data/raw/:
# Even though we're using live APIs (not manual downloads),
# saving a snapshot means: (1) we can work offline later, (2) reproducibility - someone can see exactly what data version analyzed in this project
# (3) if Yahoo Finance changes historical data or has downtime, we still have our original pull

nikkei_clean.to_csv("../data/raw/nikkei_raw.csv", index=False)
toyota_clean.to_csv("../data/raw/toyota_raw.csv", index=False)
sony_clean.to_csv("../data/raw/sony_raw.csv", index=False)

In [8]:
# WHY these 3 specific indicator codes:
# NY.GDP.MKTP.CD   -> GDP in current USD (comparable across years)
# FP.CPI.TOTL.ZG   -> Inflation rate, annual % 
# SL.UEM.TOTL.ZS   -> Unemployment rate, % of labor force
# These World Bank codes are standardized internationally - same codes work for every country


indicators = {
    'NY.GDP.MKTP.CD': 'gdp_usd',
    'FP.CPI.TOTL.ZG': 'inflation_pct',
    'SL.UEM.TOTL.ZS': 'unemployment_pct'
}

japan_macro = wb.data.DataFrame(
    list(indicators.keys()),
    economy='JPN',
    time=range(2014, 2024)
)

print(f"Shape: {japan_macro.shape}")
print(f"\nColumns: {japan_macro.columns.tolist()}")
print(f"\nIndex: {japan_macro.index.tolist()}")
japan_macro

Shape: (3, 10)

Columns: ['YR2014', 'YR2015', 'YR2016', 'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022', 'YR2023']

Index: ['FP.CPI.TOTL.ZG', 'NY.GDP.MKTP.CD', 'SL.UEM.TOTL.ZS']


,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022,YR2023
series,,,,,,,,,,
FP.CPI.TOTL.ZG,2.759227e+00,7.952796e-01,-1.272588e-01,4.841998e-01,9.890946e-01,4.687762e-01,-2.499583e-02,-2.333528e-01,2.497703e+00,3.268134e+00
NY.GDP.MKTP.CD,4.985763e+12,4.534439e+12,5.110357e+12,5.038232e+12,5.154294e+12,5.245755e+12,5.189198e+12,5.225934e+12,4.447976e+12,4.384854e+12
SL.UEM.TOTL.ZS,3.589000e+00,3.385000e+00,3.132000e+00,2.822000e+00,2.467000e+00,2.351000e+00,2.809000e+00,2.808000e+00,2.614000e+00,2.600000e+00


In [9]:
# WHY transpose (.T):
# Right now: indicators are ROWS, years are COLUMNS (wide format)
# We need: years are ROWS, indicators are COLUMNS (long format)
# This matches the structure of every other dataset in this 
# project - Data as rows, metrices as columns
# .T flips rows and columns - the simplest fix for this exact problem 

japan_macro_clean = japan_macro.T

# WHY rename columns using our indicators dict:
# Right now columns are still named 'NY.GDP.MKTP.CD' etc - 
# meaningless codes. We already built the indicators dict 
# mapping code -> readable name. Reuse it here.

japan_macro_clean = japan_macro_clean.rename(columns=indicators)


# WHY extract year as integer:
# Index values are strings like 'YR2014' right now
# We need a clean integer year column to merge with 
# everything else later (Nikkei has integer-friendly dates too)

japan_macro_clean = japan_macro_clean.reset_index()
japan_macro_clean = japan_macro_clean.rename(columns={'index': 'year_str'})
japan_macro_clean['year'] = japan_macro_clean['year_str'].str.replace('YR', '').astype(int)
japan_macro_clean = japan_macro_clean.drop(columns=['year_str'])

print(f"Shape: {japan_macro_clean.shape}")
print(f"\nColumns: {japan_macro_clean.columns.tolist()}")
japan_macro_clean

Shape: (10, 4)

Columns: ['inflation_pct', 'gdp_usd', 'unemployment_pct', 'year']


series,inflation_pct,gdp_usd,unemployment_pct,year
0,2.759227,4.985763e+12,3.589,2014
1,0.795280,4.534439e+12,3.385,2015
2,-0.127259,5.110357e+12,3.132,2016
3,0.484200,5.038232e+12,2.822,2017
4,0.989095,5.154294e+12,2.467,2018
5,0.468776,5.245755e+12,2.351,2019
6,-0.024996,5.189198e+12,2.809,2020
7,-0.233353,5.225934e+12,2.808,2021
8,2.497703,4.447976e+12,2.614,2022
9,3.268134,4.384854e+12,2.600,2023


In [10]:
japan_macro_clean.to_csv("../data/raw/japan_macro_raw.csv", index=False)
print("✓ Saved to data/raw/japan_macro_raw.csv")

✓ Saved to data/raw/japan_macro_raw.csv


In [12]:
fred = Fred(api_key=os.environ.get('FRED_API_KEY'))

# Why these 2 series specifically:
# DEXJPUS  = USD/JPY daily exchange rate 
# IRSTCB01JPM156N = BOJ short-term interest rate, monthly 
# Interest rate matters here because JapanScope's whole purpose
# is connecting MARKET behavior (Nikkei, stocks) to POLICY
# decisions (BOJ rate changes) 

usd_jpy = fred.get_series('DEXJPUS',
                          observation_start='2014-01-01',
                          observation_end='2023-12-31')

boj_rate = fred.get_series('IRSTCB01JPM156N',
                           observation_start='2014-01-01',
                           observation_end='2023-12-31')

print(f"USD/JPY shape: {usd_jpy.shape}")
print(f"BOJ rate shape: {boj_rate.shape}")
print(f"\nUSD/JPY sample:\n{usd_jpy.head()}")
print(f"\nBOJ rate sample:\n{boj_rate.head()}")

USD/JPY shape: (2608,)
BOJ rate shape: (120,)

USD/JPY sample:
2014-01-01       NaN
2014-01-02    104.84
2014-01-03    104.46
2014-01-06    104.38
2014-01-07    104.54
dtype: float64

BOJ rate sample:
2014-01-01    0.3
2014-02-01    0.3
2014-03-01    0.3
2014-04-01    0.3
2014-05-01    0.3
dtype: float64


In [13]:
# WHY convert Series to DataFrame with named columns:
# Right now usd_jpy and boj_rate are pandas Series (single column, no name). Everything else in this project is a DataFrame with named columns.

usd_jpy_df = usd_jpy.reset_index()
usd_jpy_df.columns = ['Date', 'usd_jpy']

boj_rate_df = boj_rate.reset_index()
boj_rate_df.columns = ['Date', 'boj_rate_pct']


# WHY dropna() only on usd_jpy:
# The  NaN on holidays (like Jan 1) isn't useful data -
# it's an absence of trading, not a real economic value.
# Dropping these rows keeps only actual trading days.

usd_jpy_df = usd_jpy_df.dropna()

print(f"USD/JPY cleaned shape: {usd_jpy_df.shape}")
print(f"BOJ rate shape: {boj_rate_df.shape}")

usd_jpy_df.to_csv("../data/raw/usd_jpy_raw.csv", index=False)
boj_rate_df.to_csv("../data/raw/boj_rate_raw.csv", index=False)
print("\n✓ Both saved to data/raw/")

USD/JPY cleaned shape: (2497, 2)
BOJ rate shape: (120, 2)

✓ Both saved to data/raw/
